In [2]:
from PIL import Image
import requests
import re
import os
from transformers import CLIPProcessor, CLIPModel
import glob
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["A temporal aerial collage photo with ephemeral gully formed", "A temporal aerial collage photo with no ephemeral gully formed"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/opt/conda/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The

In [22]:

def read_images_for_tile(tile_number, directory):
    images = []
    for i in range(6):
        # Construct the filename based on the tile number and image sequence
        filename = f"neg_rgb_{i}_tile_{tile_number}.jpg"
        filepath = os.path.join(directory, filename)
        
        # Check if the file exists and load the image
        if os.path.exists(filepath):
            image = Image.open(filepath)
            images.append(image)
        else:
            print(f"File {filename} does not exist.")
    
    return images

# Specify the tile number and directory containing images
tile_number = 100
directory = '/root/home/data_jpg/'  # Replace with the path to your images

# Read images for the specified tile number
images = read_images_for_tile(tile_number, directory)

def get_unique_tiles(directory):
    tile_numbers = set()
    # Regular expression to match the tile number in the filename
    pattern = r'_(\d+)\.jpg'  # Matches the tile number before .jpg

    for filename in os.listdir(directory):
        match = re.search(pattern, filename)
        if match:
            tile_number = match.group(1)
            tile_numbers.add(tile_number)

    return sorted(tile_numbers)

In [23]:
tiles = get_unique_tiles(directory)

In [21]:
inputs = processor(text=["an aerial photo of a location with an ephemeral gully formed", "an aerial photo of a location with no ephemeral gully formed"], images=images, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities
print(probs)
# Determine if any image has a high probability for the gully class
gully_detected = (probs[:, 0] > 0.5).any().item()  # Check if any image has gully class probability > 0.5

if gully_detected:
    print("At least one image has an ephemeral gully.")
else:
    print("No ephemeral gully detected in any of the images.")

tile_number = 100
directory = '/root/home/data_jpg/'  # Replace with the path to your images

# Read images for the specified tile number
images = read_images_for_tile(tile_number, directory)

tensor([[0.2854, 0.7146],
        [0.3014, 0.6986],
        [0.1831, 0.8169],
        [0.1297, 0.8703],
        [0.2861, 0.7139],
        [0.1992, 0.8008]], grad_fn=<SoftmaxBackward0>)
No ephemeral gully detected in any of the images.


In [19]:
probs

tensor([[0.8889, 0.1111],
        [0.9538, 0.0462],
        [0.8749, 0.1251],
        [0.7142, 0.2858],
        [0.9312, 0.0688],
        [0.8492, 0.1508]], grad_fn=<SoftmaxBackward0>)

In [4]:

data_dir = '/root/home/data/'


# List all .jpg files in the specified directory (non-recursive)
jpg_files = glob.glob(f"{data_dir}/*.jpg")
#print(jpg_files)


In [11]:
device = "cuda"
model.to(device)

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [24]:
import tqdm
import torch
i=0
n_files = len(jpg_files)
clip_results = {}
gullies=0
# A temporal aerial collage photo with an ephemeral gully formed
for image_file in jpg_files:
    print(f"Processing -- {os.path.basename(image_file)} -- {i}/{n_files}")
    img = Image.open(image_file)
    inputs = processor(text=["A photo of an ephemeral gully", "A photo with no ephemeral gully"], images=img, return_tensors="pt", padding=True)
    inputs.to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
    probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities
    #print(probs)
    
    # Determine if any image has a high probability for the gully class
    gully_detected = (probs[:, 0] > 0.5).any().item()  # Check if any image has gully class probability > 0.5
    #print(gully_detected)
    if gully_detected :
        gullies+=1
        #print(f"Gully present: {gullies}")
        clip_results[os.path.basename(image_file)]=1
    else :
        clip_results[os.path.basename(image_file)]=0
    i+=1
    del inputs
    

Processing -- neg_collage_100.jpg -- 0/900
Processing -- neg_collage_1000.jpg -- 1/900
Processing -- neg_collage_1001.jpg -- 2/900
Processing -- neg_collage_1002.jpg -- 3/900
Processing -- neg_collage_1003.jpg -- 4/900
Processing -- neg_collage_1004.jpg -- 5/900
Processing -- neg_collage_1005.jpg -- 6/900
Processing -- neg_collage_1006.jpg -- 7/900
Processing -- neg_collage_1007.jpg -- 8/900
Processing -- neg_collage_1009.jpg -- 9/900
Processing -- neg_collage_101.jpg -- 10/900
Processing -- neg_collage_1010.jpg -- 11/900
Processing -- neg_collage_1011.jpg -- 12/900
Processing -- neg_collage_1013.jpg -- 13/900
Processing -- neg_collage_1014.jpg -- 14/900
Processing -- neg_collage_1015.jpg -- 15/900
Processing -- neg_collage_1017.jpg -- 16/900
Processing -- neg_collage_1018.jpg -- 17/900
Processing -- neg_collage_1019.jpg -- 18/900
Processing -- neg_collage_102.jpg -- 19/900
Processing -- neg_collage_1020.jpg -- 20/900
Processing -- neg_collage_1021.jpg -- 21/900
Processing -- neg_colla

Processing -- neg_collage_276.jpg -- 184/900
Processing -- neg_collage_277.jpg -- 185/900
Processing -- neg_collage_279.jpg -- 186/900
Processing -- neg_collage_281.jpg -- 187/900
Processing -- neg_collage_282.jpg -- 188/900
Processing -- neg_collage_283.jpg -- 189/900
Processing -- neg_collage_285.jpg -- 190/900
Processing -- neg_collage_286.jpg -- 191/900
Processing -- neg_collage_287.jpg -- 192/900
Processing -- neg_collage_288.jpg -- 193/900
Processing -- neg_collage_289.jpg -- 194/900
Processing -- neg_collage_290.jpg -- 195/900
Processing -- neg_collage_291.jpg -- 196/900
Processing -- neg_collage_292.jpg -- 197/900
Processing -- neg_collage_293.jpg -- 198/900
Processing -- neg_collage_294.jpg -- 199/900
Processing -- neg_collage_295.jpg -- 200/900
Processing -- neg_collage_296.jpg -- 201/900
Processing -- neg_collage_297.jpg -- 202/900
Processing -- neg_collage_298.jpg -- 203/900
Processing -- neg_collage_299.jpg -- 204/900
Processing -- neg_collage_300.jpg -- 205/900
Processing

Processing -- neg_collage_777.jpg -- 368/900
Processing -- neg_collage_778.jpg -- 369/900
Processing -- neg_collage_779.jpg -- 370/900
Processing -- neg_collage_78.jpg -- 371/900
Processing -- neg_collage_780.jpg -- 372/900
Processing -- neg_collage_782.jpg -- 373/900
Processing -- neg_collage_783.jpg -- 374/900
Processing -- neg_collage_784.jpg -- 375/900
Processing -- neg_collage_786.jpg -- 376/900
Processing -- neg_collage_787.jpg -- 377/900
Processing -- neg_collage_788.jpg -- 378/900
Processing -- neg_collage_789.jpg -- 379/900
Processing -- neg_collage_79.jpg -- 380/900
Processing -- neg_collage_790.jpg -- 381/900
Processing -- neg_collage_793.jpg -- 382/900
Processing -- neg_collage_794.jpg -- 383/900
Processing -- neg_collage_795.jpg -- 384/900
Processing -- neg_collage_796.jpg -- 385/900
Processing -- neg_collage_797.jpg -- 386/900
Processing -- neg_collage_798.jpg -- 387/900
Processing -- neg_collage_80.jpg -- 388/900
Processing -- neg_collage_801.jpg -- 389/900
Processing --

Processing -- pos_collage_32.jpg -- 552/900
Processing -- pos_collage_33.jpg -- 553/900
Processing -- pos_collage_34.jpg -- 554/900
Processing -- pos_collage_349.jpg -- 555/900
Processing -- pos_collage_350.jpg -- 556/900
Processing -- pos_collage_352.jpg -- 557/900
Processing -- pos_collage_355.jpg -- 558/900
Processing -- pos_collage_356.jpg -- 559/900
Processing -- pos_collage_357.jpg -- 560/900
Processing -- pos_collage_358.jpg -- 561/900
Processing -- pos_collage_359.jpg -- 562/900
Processing -- pos_collage_36.jpg -- 563/900
Processing -- pos_collage_360.jpg -- 564/900
Processing -- pos_collage_361.jpg -- 565/900
Processing -- pos_collage_362.jpg -- 566/900
Processing -- pos_collage_364.jpg -- 567/900
Processing -- pos_collage_365.jpg -- 568/900
Processing -- pos_collage_366.jpg -- 569/900
Processing -- pos_collage_367.jpg -- 570/900
Processing -- pos_collage_368.jpg -- 571/900
Processing -- pos_collage_369.jpg -- 572/900
Processing -- pos_collage_371.jpg -- 573/900
Processing -- 

Processing -- pos_collage_647.jpg -- 736/900
Processing -- pos_collage_649.jpg -- 737/900
Processing -- pos_collage_650.jpg -- 738/900
Processing -- pos_collage_651.jpg -- 739/900
Processing -- pos_collage_653.jpg -- 740/900
Processing -- pos_collage_654.jpg -- 741/900
Processing -- pos_collage_656.jpg -- 742/900
Processing -- pos_collage_657.jpg -- 743/900
Processing -- pos_collage_658.jpg -- 744/900
Processing -- pos_collage_661.jpg -- 745/900
Processing -- pos_collage_662.jpg -- 746/900
Processing -- pos_collage_663.jpg -- 747/900
Processing -- pos_collage_664.jpg -- 748/900
Processing -- pos_collage_666.jpg -- 749/900
Processing -- pos_collage_667.jpg -- 750/900
Processing -- pos_collage_668.jpg -- 751/900
Processing -- pos_collage_669.jpg -- 752/900
Processing -- pos_collage_67.jpg -- 753/900
Processing -- pos_collage_670.jpg -- 754/900
Processing -- pos_collage_671.jpg -- 755/900
Processing -- pos_collage_676.jpg -- 756/900
Processing -- pos_collage_677.jpg -- 757/900
Processing 

In [25]:
gullies

570